# Final project deliverable

## 1.Modeling and solution methods:


### 1.1 Data formalization : 
The Graph: The network is a complete directed graph $G = (V, A)$ where $V$ is the set of nodes (the depot $0$ plus the customers $C$). 
<br>
The Fleets: A set of homogeneous vehicles $K$, each with a maximum capacity $Q$. 
<br>
The Customers: Each customer $i \in C$ has a demand $q_i$, a service time $s_i$, and a strict time window $[e_i, l_i]$ (start and end times). 
<br>
Edges: Each edge $(i, j) \in A$ has a travel cost $c_{ij}$ (distance) and a travel time $t_{ij}$.

### 1.2 Complexity Proof
The CVRPTW is strictly **NP-Hard**. It is a generalization of two well-known NP-Hard problems: 
1. The **Traveling Salesperson Problem (TSP)** (finding the shortest path).
2. The **Bin Packing Problem** (fitting demands $q_i$ into capacities $Q$).
Because the solution space grows factorially $\mathcal{O}(n!)$, finding an exact mathematical optimum becomes computationally intractable as the number of customers increases.
To more precision cf. to first deliverable.

### 1.3 Integer Linear Program (ILP) Formulation
To formally model the problem, we define two decision variables:
- $x_{ijk} \in \{0, 1\}$ : equals $1$ if vehicle $k$ travels directly from node $i$ to node $j$, and $0$ otherwise.
- $l_{ik} \ge 0$ : the arrival time of vehicle $k$ at node $i$.

**Objective Function:**
Minimize the total travel cost (or time):
$$ \min \sum_{k \in K} \sum_{i \in V} \sum_{j \in V} c_{ij} x_{ijk} $$

### 1.4 Potential Applications of the Exact Approach
Due to combinatorial explosion, solving this ILP model using exact solvers fails due to memory saturation for instances with more than 15-20 customers. However, the exact approach remains highly valuable for:
- **Solving small instances** directly .
- **Establishing Benchmarks:** Providing the absolute mathematical optimum on small datasets to test the accuracy of our heuristics.
- **Calculating Lower/Upper Bounds:** Using linear relaxation (allowing $x_{ijk}$ to be decimal) to calculate mathematical bounds, helping to evaluate the "Gap" of metaheuristic solutions on large datasets.

### Intensification vs. Diversification: The Core Trade-off

The performance of any metaheuristic depends entirely on its ability to balance two opposing forces: **Intensification** (Exploitation) and **Diversification** (Exploration). Understanding this duality is key to analyzing why certain algorithms fail or succeed on complex instances.

#### 1. Intensification (Exploitation)
- **Concept:** This is the act of focusing the search within a specific, promising region of the solution space. The goal is to dig as deep as possible into a local area to find the absolute best solution it contains (the local optimum). 
- **Analogy:** Using a magnifying glass to find the biggest gold nugget in a single cave.
- **Algorithms in our project:**
  - **Hill Climbing:** Pure intensification. It greedily takes the best immediate neighbor and stops as soon as it hits a wall. It has zero diversification.
  - **VND (Variable Neighborhood Descent):** A "super-intensification" tool. By switching between different operators (Relocate, Swap, 2-Opt), it digs deeper than Hill Climbing, but it still strictly descends and never explores new, worse regions.
  - **Nearest Neighbor:** A purely intensive constructive heuristic. It only looks at the immediate shortest distance, ignoring the global picture.

#### 2. Diversification (Exploration)
- **Concept:** This is the act of forcing the algorithm to leave its current region to explore entirely new, unvisited, or diverse areas of the global search space. It prevents the algorithm from getting permanently stuck in a local optimum.
- **Analogy:** Using a radar to scan the entire mountain range to find new caves that might hold even more gold.
- **How our Metaheuristics manage the balance:**
  - **Simulated Annealing (Dynamic Balance):** 
    It manages the trade-off through *Time* (the cooling schedule). At the beginning (high temperature), it frequently accepts bad moves: this is pure **diversification** (bouncing around the search space). At the end (temperature near zero), it only accepts improvements: it switches to pure **intensification** (acting like Hill Climbing).
  - **Tabu Search (Memory-driven Balance):** 
    It intensifies by always selecting the best available neighbor. However, it forces **diversification** through its Tabu List. By forbidding the algorithm from reversing its recent moves, it acts like a wall that pushes the search trajectory out of the local valley and into unexplored territories.
  - **Genetic Algorithms (Population-driven Balance):**
    If we were to use a Genetic Algorithm, the balance is achieved through its operators. *Selection* and *Crossover* (combining good parents) drive **intensification** by refining good genetic traits. Conversely, *Mutation* (randomly altering a gene) injects pure **diversification**, ensuring new genetic material is explored so the population doesn't prematurely converge into clones.



#### Data set cf. GitHub : https://github.com/PyVRP/VRPLIB

## 2. Implementation

### 2.1 Modelisation of the Network


In [ ]:
import random
from collections import deque

class Node:
    """
    Mathematical modeling of each node (vertex) in the network, whether it is a warehouse or a customer.
    Each customer becomes an “Object” with its own attributes, derived directly from the constraint variables. 
    """
    def __init__(self, node_id, demand=0, earliest_time=0, latest_time=float('inf'), service_time=0):
        self.node_id = node_id  # Indice i
        self.q = demand         # Demande q_i
        self.e = earliest_time  # Ouverture de la fenêtre e_i
        self.l = latest_time    # Fermeture de la fenêtre l_i
        self.s = service_time   # Temps de service s_i

class CVRPTW_Network:
    """
    Models the complete directed graph G = (V, E) and centralizes the problem's parameters.
    Avoids passing multiple lists to the algorithms. This unique object acts as 
    the mathematical environment in which the metaheuristics evolve.
    """
    def __init__(self, n_customers, k_trucks, capacity):
        self.N = n_customers    # Nombre total de clients N
        self.K = k_trucks       # Nombre de camions K
        self.Q = capacity       # Capacité maximale d'un camion Q
        self.B = float('inf')   # Borne de coût B'

        self.V = list(range(self.N + 1))
        self.C = [node for node in self.V if node != 0]
        self.E = [(i, j) for i in self.V for j in self.V if i != j]

        self.nodes = {}
        self.travel_times = {}

    def add_node(self, node_id, demand=0, e=0, l=float('inf'), s=0):
        """Instantiates and adds a vertex to the graph."""
        self.nodes[node_id] = Node(node_id, demand, e, l, s)

    def set_travel_time(self, i, j, time_val):
        """Fills in the time/cost matrix (the formal equivalent of the variable c_ij)."""
        if (i, j) in self.E:
            self.travel_times[(i, j)] = time_val

    def calculate_objective(self, routes):
        """
        Calculates the objective function Z of the Integer Linear Programming (ILP) model, where Z = Sum(C_ij * X_ij).
        """
        total_time = 0
        for route in routes:
            for idx in range(len(route) - 1):
                i = route[idx]
                j = route[idx + 1]
                total_time += self.travel_times.get((i, j), 0)
        return total_time

In [ ]:
def charger_instance_cvrptw(chemin_fichier, limite_clients=None):
    """
    Reads standard files from the literature (Solomon format) and converts
    the raw data into objects in our formal model (CVRPTW_Network and Node).
    """
    import vrplib
    data = vrplib.read_instance(chemin_fichier, instance_format="solomon") 
    #print(data)
    # Instance Size Management
    dimension_totale = len(data['demand'])
    if limite_clients is not None:
        dimension = min(limite_clients + 1, dimension_totale)
    else:
        dimension = dimension_totale
        
    n_clients = dimension - 1 
    capacite = data['capacity']
    nb_camions = data.get('vehicles', 25) 
    
    # Network Creation
    reseau = CVRPTW_Network(n_customers=n_clients, k_trucks=nb_camions, capacity=capacite)
    
    for i in range(dimension):
        reseau.add_node(
            node_id=i,
            demand=data['demand'][i],
            e=data['time_window'][i][0],
            l=data['time_window'][i][1],
            s=data['service_time'][i]
        )
        
    distances = data['edge_weight']
    for i in reseau.V:
        for j in reseau.V:
            if i != j:
                reseau.set_travel_time(i, j, distances[i][j])
                
    return reseau

## Verification of Time Windows

In [ ]:
def est_realisable(route_tuple, reseau):
    """
    Checks the feasibility of a candidate route of window time.
    """
    charge_actuelle = 0
    temps_actuel = 0
    
    for k in range(len(route_tuple) - 1):
        i = route_tuple[k]
        j = route_tuple[k+1]
        
        charge_actuelle += reseau.nodes[i].q
        if charge_actuelle > reseau.Q:
            return False
            
        temps_actuel = max(temps_actuel, reseau.nodes[i].e)
        temps_actuel += reseau.nodes[i].s + reseau.travel_times.get((i, j), float('inf'))
        
        if temps_actuel > reseau.nodes[j].l:
            return False 
            
    return True

In [ ]:
def solution_est_realisable(solution, reseau):
    """Vérificateur global : vérifie si TOUTES les routes de la solution sont valides."""
    for route in solution:
        if not est_realisable(route, reseau):
            return False
    return True

## 2. Neighborhood Structure and Local Search Operators

In the context of metaheuristics (Tabu Search, Simulated Annealing) and descent heuristics (VND), the algorithm navigates through a **search space** by transitioning from a solution $S$ to a neighboring solution $S'$. 
A "neighbor" is defined as a solution generated by applying **a single mathematical modification** (a move) to the current solution.

For our CVRPTW problem (multi-route), we have modeled and implemented three complementary neighborhood operators. This diversity is crucial (especially for the VND algorithm) to avoid getting trapped in a local optimum.

### 1. The SWAP Operator (Exchange)
- **Principle:** This operator selects two customers $i$ and $j$ in the current solution and simply swaps their positions.
- **Scope (Intra- and Inter-route):** The two swapped customers can belong to the same truck's route, or to two different trucks.
- **Logistical benefit:** This move is highly effective for balancing the load between two vehicles or for correcting assignments if two geographically close customers were mistakenly assigned to opposite routes.

### 2. The RELOCATE Operator (Displacement / Insertion)
- **Principle:** This operator extracts a customer $i$ from their current position and inserts them at a new position $j$ (which naturally shifts the other customers).
- **Scope (Intra- and Inter-route):** The customer can be moved further along within their own truck's route, or transferred to an entirely different truck.
- **Specificity of our implementation (Stochastic Approach):** 
  In an instance like C101 (100 customers), the number of possible insertions is colossal. To prevent an explosion in computation time when evaluating the neighborhood, we implemented a **"FAST Relocate"** version. Instead of generating exhaustive combinations, the operator performs a random (stochastic) sampling capped at a maximum number of neighbors (e.g., 150). This reduces the iteration time from several minutes down to a few seconds, while maintaining an excellent probability of finding an improving move.

### 3. The 2-OPT Operator (Segment Reversal)
- **Principle:** A classic operator from the Traveling Salesperson Problem (TSP). It involves selecting a segment (a subsequence) of customers within a route, and completely reversing their order of visit (e.g., `A -> B -> C -> D` becomes `A -> C -> B -> D`).
- **Scope (Strictly Intra-route):** This move applies solely within a single truck's route. It does not alter the assignment of customers to vehicles.
- **Logistical benefit:** Geometrically, 2-OPT is the perfect tool for "uncrossing" a route. If a truck's path forms a bow-tie shape (which is sub-optimal in terms of actual distance), reversing a segment untangles the path and drastically shortens the traveled distance, without affecting the truck's load.

In [ ]:

import random
def voisinage_multi_swap(solution):
    """
    Method 1: SWAP (Exchange)
    Swaps the positions of two customers. The swap can occur within the same route 
    (intra-route) or between two different trucks (inter-route).
    """
    solution_list = [list(r) for r in solution]
    nb_routes = len(solution_list)
    
    for r1 in range(nb_routes):
        for i in range(1, len(solution_list[r1]) - 1): # Ignore les dépots (0)
            for r2 in range(r1, nb_routes):
                start_j = i + 1 if r1 == r2 else 1
                for j in range(start_j, len(solution_list[r2]) - 1):
                    voisin = [list(r) for r in solution_list]
                    # Échange des deux clients
                    voisin[r1][i], voisin[r2][j] = voisin[r2][j], voisin[r1][i]
                    
                    yield tuple(tuple(r) for r in voisin)



def voisinage_multi_relocate(solution, max_voisins=150):
    """
    Method 2: RELOCATE (Insertion) - FAST Stochastic Version
    Extracts a customer and inserts them at a new position.
    Uses random sampling (stochastic approach) to avoid computational explosion, 
    reducing generation time from 8 minutes to ~10 seconds on large instances.
    """
    solution_list = [list(r) for r in solution]
    nb_routes = len(solution_list)
    voisins_generes = 0
    essais = 0
    
    # On limite la recherche à 'max_voisins' pour être ultra-rapide
    while voisins_generes < max_voisins and essais < max_voisins * 10:
        essais += 1
        
        # Tirage d'un client au hasard
        r1 = random.randint(0, nb_routes - 1)
        if len(solution_list[r1]) <= 2: continue # On ignore les camions vides
        i = random.randint(1, len(solution_list[r1]) - 2)
        client = solution_list[r1][i]
        
        # Tirage d'une destination au hasard
        r2 = random.randint(0, nb_routes - 1)
        j = random.randint(1, len(solution_list[r2]) - 1)
        
        if r1 == r2 and i == j: continue
            
        # Création du voisin
        voisin = [list(r) for r in solution_list]
        voisin[r1].pop(i)
        
        if r1 == r2 and j > i:
            voisin[r2].insert(j-1, client)
        else:
            voisin[r2].insert(j, client)
            
        voisin_propre = tuple(tuple(r) for r in voisin if len(r) > 2)
        
        voisins_generes += 1
        yield voisin_propre


def voisinage_multi_2opt(solution):
    """
    Method 3: 2-OPT (Segment Reversal)
    Completely reverses the sequence of a subset of customers within a single truck.
    Strictly an intra-route operator. Highly effective for uncrossing overlapping routes.
    """
    solution_list = [list(r) for r in solution]
    nb_routes = len(solution_list)
    
    for r in range(nb_routes):
        route = solution_list[r]
        n = len(route)
        for i in range(1, n - 2):
            for j in range(i + 1, n - 1):
                voisin = [list(rt) for rt in solution_list]
                # Inversion du segment [i:j]
                voisin[r][i:j+1] = reversed(voisin[r][i:j+1])
                
                yield tuple(tuple(rt) for rt in voisin)



def voisinage_multi(solution, methode="relocate"):
    """Sélecteur global pour générer le voisinage selon la méthode choisie."""
    
    if methode == "random":
        methode = random.choice(["swap", "relocate", "2opt"])
    if methode == "swap":
        return voisinage_multi_swap(solution)
    elif methode == "relocate":
        return voisinage_multi_relocate(solution)
    elif methode == "2opt":
        return voisinage_multi_2opt(solution)
    else:
        raise ValueError(f"Méthode '{methode}' inconnue. Choisissez 'swap', 'relocate' ou '2opt'.")

## 4. The Multi-Start Principle (and Why We Excluded It)

**Principle:** 
The Multi-Start approach is a global exploration strategy commonly used in optimization. Instead of running a metaheuristic (like Tabu Search or Simulated Annealing) only once, the algorithm is restarted multiple times from completely different initial solutions (often randomly generated). Once all the independent runs are completed, the algorithm returns the absolute best solution found across all attempts. The main advantage is that it forces the algorithm to explore entirely different regions of the search space, drastically reducing the risk of getting permanently trapped in a single local optimum.

**Why we chose not to implement it:**
While theoretically powerful, we deliberately excluded the Multi-Start strategy from our final implementation due to its prohibitive computational cost. 

In the highly constrained context of the CVRPTW, especially on large-scale industrial instances like C101 (100 customers), checking the strict capacity and time-window constraints for thousands of neighboring solutions is computationally expensive. Running a deep metaheuristic (e.g., Tabu Search with 5,000 iterations) already requires significant processing time. 

If we were to use a Multi-Start approach with 20 or 50 different starting points, the total execution time would be multiplied by 20 or 50, making the tool irrelevant for daily operational use where logistics dispatchers need fast routing decisions. 

**Our alternative:**
Rather than relying on brute-force repetitions from poor random starting points, we opted for an **intelligent hybridization strategy**. We generate a single, highly optimized initial solution using a constructive heuristic (Nearest Neighbor), and dedicate 100% of our computational budget to deeply improving this specific solution via local search and metaheuristics.

### 1. Heuristics
**Definition:** A heuristic is a problem-dependent algorithmic method designed to find a "good enough" feasible solution quickly, without guaranteeing the absolute mathematical optimum. 
**Characteristics:** 
- They are usually deterministic and "greedy" (short-sighted). 
- They strictly seek immediate improvement and never accept a degradation of the solution's cost.
- Consequently, they systematically get trapped in the first **local optimum** they encounter.
- **In our project:** We use constructive heuristics (like the *Nearest Neighbor* to build the initial routing plan) and local search heuristics (like *Hill Climbing* and *Variable Neighborhood Descent - VND* to refine it).

In [ ]:
def nearest_neighbor_solution_multi(reseau):
    """
    Heuristique Constructive (Multi-Routes) : Plus Proche Voisin.
    Génère une flotte de camions. Dès qu'un camion est plein ou hors délai, 
    il rentre au dépôt et un nouveau camion prend le relais.
    """
    clients_non_visites = set(reseau.C)
    routes = []
    
    # Tant qu'il reste des clients à livrer
    while clients_non_visites:
        route_courante = [0] # Nouveau camion part du dépôt
        temps_actuel = 0
        charge_actuelle = 0
        camion_peut_continuer = True
        
        while camion_peut_continuer and clients_non_visites:
            client_actuel = route_courante[-1]
            meilleur_prochain = None
            meilleur_cout = float('inf')
            meilleur_temps_arrivee = 0
            
            for candidat in clients_non_visites:
                # 1. Vérification de la Capacité Q
                if charge_actuelle + reseau.nodes[candidat].q > reseau.Q:
                    continue
                    
                t_trajet = reseau.travel_times.get((client_actuel, candidat), float('inf'))
                t_arrivee = temps_actuel + t_trajet
                
                # 2. Gestion de l'heure d'arrivée (attente si en avance)
                if t_arrivee < reseau.nodes[candidat].e:
                    t_arrivee = reseau.nodes[candidat].e
                    
                # 3. Vérification de la Fenêtre de Temps
                if t_arrivee <= reseau.nodes[candidat].l:
                    cout = t_arrivee - temps_actuel 
                    if cout < meilleur_cout:
                        meilleur_cout = cout
                        meilleur_prochain = candidat
                        meilleur_temps_arrivee = t_arrivee
            
            # Ajout du client trouvé
            if meilleur_prochain is not None:
                route_courante.append(meilleur_prochain)
                clients_non_visites.remove(meilleur_prochain)
                temps_actuel = meilleur_temps_arrivee + reseau.nodes[meilleur_prochain].s
                charge_actuelle += reseau.nodes[meilleur_prochain].q
            else:
                # Aucun client ne peut être ajouté à ce camion
                camion_peut_continuer = False 
                
        # Le camion rentre au dépôt
        route_courante.append(0)
        
        # Sécurité : Si un camion part et revient à vide (0, 0) alors qu'il reste des clients
        if len(route_courante) == 2 and clients_non_visites:
            raise Exception(f"Impossible de livrer les clients restants : {clients_non_visites} (Contraintes trop strictes)")
            
        routes.append(tuple(route_courante))
        
    return routes

### Hills Climbings
The current node is initialized to its initial state. Iteratively, the current node is compared to its immediate successors (neighbors). <br>The best immediate successor—that is, the one with the highest value compared to the current node—becomes the new current node. If no such neighbor exists, the process stops, and the current node is returned as the solution.

In [70]:
def hill_climbing_multi(solution_initiale, reseau, methode_voisinage="relocate"):
    solution_courante = solution_initiale
    nouveau = True
    
    while nouveau:
        meilleure_solution = solution_courante
        meilleur_score = reseau.calculate_objective(solution_courante)
        
        # On utilise ici l'argument methode_voisinage
        for voisin in voisinage_multi(solution_courante, methode=methode_voisinage):
            if solution_est_realisable(voisin, reseau):
                score_voisin = reseau.calculate_objective(voisin)
                if score_voisin < meilleur_score:
                    meilleure_solution = voisin
                    meilleur_score = score_voisin
                    
        nouveau = (meilleure_solution != solution_courante)
        solution_courante = meilleure_solution
        
    return solution_courante

In [71]:
def vnd_multi(solution_initiale, reseau):
    """
    Descente à Voisinage Variable (VND).
    Combine vos 3 opérateurs : si l'un bloque, il passe au suivant pour s'échapper.
    Ordre typique : 1. Relocate (précis), 2. Swap (intermédiaire), 3. 2-opt (lourd).
    """
    # L'ordre d'exploration de vos voisinages
    methodes = ["relocate", "swap", "2opt"]
    
    solution_courante = solution_initiale
    meilleur_score = reseau.calculate_objective(solution_courante)
    
    k = 0 # Index de la méthode actuelle
    
    while k < len(methodes):
        methode_actuelle = methodes[k]
        
        # On lance un Hill Climbing complet jusqu'à être bloqué avec CETTE méthode
        sol_hc = hill_climbing_multi(solution_courante, reseau, methode_voisinage=methode_actuelle)
        score_hc = reseau.calculate_objective(sol_hc)
        
        if score_hc < meilleur_score:
            # L'algorithme a réussi à améliorer le score !
            solution_courante = sol_hc
            meilleur_score = score_hc
            
            # Règle d'or du VND : dès qu'on s'améliore, on retourne à la méthode la plus fine
            k = 0 
        else:
            # Bloqué ! La méthode actuelle ne donne plus rien.
            # On passe à la méthode suivante (ex: on lâche le relocate pour tenter un swap)
            k += 1 
            
    return solution_courante

### 2. Metaheuristics
**Definition:** A metaheuristic (from the Greek *meta*, meaning "beyond" or "higher level") is a generic, overarching algorithmic framework designed to guide and control underlying heuristics. 
**Characteristics:**
- They are problem-independent (the same core logic can be applied to vehicle routing, employee scheduling, or financial portfolios).
- Their main purpose is to **escape local optima**. To achieve this, they use advanced mechanisms (such as memory, probabilities, or physical analogies) that allow them to temporarily accept a *worse* solution to explore new regions of the global search space.
- **In our project:** We implemented *Tabu Search* (which uses a short-term memory to forbid recently visited moves) and *Simulated Annealing* (which uses a temperature-based probability to accept worse moves) to break through the local optima where our simple heuristics got stuck.

### Tabu Search

In [72]:
from collections import deque

def recherche_tabou_multi(solution_initiale, reseau, taille_tabou, iter_max, methode_voisinage="relocate"):
    """
    Métaheuristique Tabou compatible avec plusieurs camions et les Datasets VRPLIB.
    """
    liste_tabou = deque(maxlen=taille_tabou)
    solution_courante = solution_initiale
    meilleure_globale = solution_initiale
    
    # APPEL CORRIGÉ : On passe la solution directement
    valeur_meilleure_globale = reseau.calculate_objective(solution_initiale)
    nb_iter_sans_amelioration = 0
    
    while nb_iter_sans_amelioration < iter_max:
        meilleur_voisin = None
        valeur_meilleur_voisin = float('inf')
        
        # APPEL CORRIGÉ : On utilise le sélecteur multi-voisinages
        for voisin in voisinage_multi(solution_courante, methode=methode_voisinage):
            
            # APPEL CORRIGÉ : On vérifie TOUS les camions avec solution_est_realisable
            if voisin not in liste_tabou and solution_est_realisable(voisin, reseau):
                score_voisin = reseau.calculate_objective(voisin)
                
                if score_voisin < valeur_meilleur_voisin:
                    meilleur_voisin = voisin
                    valeur_meilleur_voisin = score_voisin
        
        # Si aucun voisin n'est valide (cas très rare mais possible si très contraint)
        if meilleur_voisin is None:
            break
            
        # Mise à jour des records
        if valeur_meilleur_voisin < valeur_meilleure_globale:
            meilleure_globale = meilleur_voisin
            valeur_meilleure_globale = valeur_meilleur_voisin
            nb_iter_sans_amelioration = 0
        else:
            nb_iter_sans_amelioration += 1
            
        # On avance
        solution_courante = meilleur_voisin
        liste_tabou.append(solution_courante)
        
    return meilleure_globale

### Simulated Annealing

In [73]:
import math
import random

def recuit_simule_multi(solution_initiale, reseau, temp_initiale, taux_refroidissement, iter_max, methode_voisinage="relocate"):
    """
    Métaheuristique : Recuit Simulé (Simulated Annealing) compatible avec plusieurs camions.
    Accepte des détours selon une probabilité qui baisse avec le temps.
    """
    solution_courante = solution_initiale
    meilleure_globale = solution_initiale
    
    # APPEL CORRIGÉ : On calcule le coût en passant directement le tuple de routes
    valeur_courante = reseau.calculate_objective(solution_courante)
    valeur_meilleure_globale = valeur_courante
    
    temperature = temp_initiale
    
    for _ in range(iter_max):
        # APPEL CORRIGÉ : Voisinage multi-routes et vérification globale de la validité
        voisins_valides = [
            v for v in voisinage_multi(solution_courante, methode=methode_voisinage) 
            if solution_est_realisable(v, reseau)
        ]
        
        if not voisins_valides:
            break # Sécurité : aucun voisin possible sans violer les contraintes (poids, temps)
            
        # 1. On tire UN SEUL voisin au hasard
        voisin_choisi = random.choice(voisins_valides)
        valeur_voisin = reseau.calculate_objective(voisin_choisi)
        
        # 2. Calcul de la différence de score
        # delta < 0 : le voisin est plus rapide (amélioration)
        # delta > 0 : le voisin est plus lent (dégradation)
        delta = valeur_voisin - valeur_courante
        
        # 3. Critère d'acceptation (Metropolis)
        if delta < 0:
            # C'est meilleur : on accepte d'office l'amélioration
            solution_courante = voisin_choisi
            valeur_courante = valeur_voisin
        else:
            # C'est pire : on calcule la probabilité d'accepter ce détour
            if temperature > 0.001:
                # Formule mathématique d'acceptation
                probabilite = math.exp(-delta / temperature)
                
                # Le coup de dés
                if random.random() < probabilite:
                    solution_courante = voisin_choisi
                    valeur_courante = valeur_voisin
                    
        # 4. Mise à jour du record absolu
        if valeur_courante < valeur_meilleure_globale:
            meilleure_globale = solution_courante
            valeur_meilleure_globale = valeur_courante
            
        # 5. On refroidit la température
        temperature *= taux_refroidissement
        
    return meilleure_globale

## 3.Experimental Study

In [ ]:
import os
import csv
import time
import random
import vrplib
import numpy as np

# ==========================================
# 1. PARAMÈTRES DU BENCHMARK STATISTIQUE
# ==========================================
FOLDER_DATA = "data"
FILE_RUNS_CSV = "results/benchmark_runs_detailles.csv"
FILE_STATS_CSV = "results/benchmark_stats_agregees.csv"

# Nombre d'exécutions par instance (augmentez à 20 ou 50 pour le rendu final)
N_SEEDS = 2

noms_algos = [
    'Aléatoire',
    'Plus Proche Voisin',
    'Hill Climbing',
    'VND',
    'Tabou Search',
    'Simulated Annealing',
    'Ant Colony (ACO)',
    'Algo Génétique'
]

# ==========================================
# 2. SÉLECTION MULTIPLE DES FICHIERS
# ==========================================
print("=" * 65)
print(" 🚀 BENCHMARK STATISTIQUE MULTI-INSTANCES (CVRPTW)")
print("=" * 65)

if not os.path.exists(FOLDER_DATA):
    raise FileNotFoundError(f"Le dossier '{FOLDER_DATA}' est introuvable.")

choix = input("Entrez les instances séparées par une virgule (ex: C101, C102, C201) ou 'ALL' : ").strip()

if choix.upper() == "ALL":
    instances_a_tester = sorted([
        os.path.splitext(f)[0] for f in os.listdir(FOLDER_DATA)
        if f.endswith(".txt") and os.path.exists(os.path.join(FOLDER_DATA, os.path.splitext(f)[0] + ".sol"))
    ])
else:
    instances_a_tester = [inst.strip() for inst in choix.split(",") if inst.strip()]

print(f"\n📋 Instances sélectionnées ({len(instances_a_tester)}) : {instances_a_tester}")
print(f"🎲 Nombre de seeds par instance : {N_SEEDS}\n")

# Préparation des fichiers CSV
os.makedirs("results", exist_ok=True)

with open(FILE_RUNS_CSV, mode='w', newline='', encoding='utf-8') as f_runs, \
     open(FILE_STATS_CSV, mode='w', newline='', encoding='utf-8') as f_stats:
    
    writer_runs = csv.writer(f_runs)
    writer_runs.writerow([
        "Instance", "Seed", "Algorithm", "Score", "Optimal_Score", 
        "Gap_Pct", "Trucks", "Optimal_Trucks", "Time_s"
    ])
    
    writer_stats = csv.writer(f_stats)
    writer_stats.writerow([
        "Instance", "Algorithm", "N_Seeds", "Mean_Score", "Std_Score", 
        "Optimal_Score", "Mean_Gap_Pct", "Std_Gap_Pct", "Mean_Trucks", 
        "Optimal_Trucks", "Mean_Time_s", "Std_Time_s"
    ])

    # ==========================================
    # 3. BOUCLE PRINCIPALE SUR LES INSTANCES
    # ==========================================
    for instance_name in instances_a_tester:
        txt_file = os.path.join(FOLDER_DATA, f"{instance_name}.txt")
        sol_file = os.path.join(FOLDER_DATA, f"{instance_name}.sol")
        
        if not os.path.exists(txt_file) or not os.path.exists(sol_file):
            print(f"⚠️ Fichiers manquants pour '{instance_name}' (.txt ou .sol). Instance ignorée.")
            continue
            
        print("=" * 65)
        print(f" 📂 TRAITEMENT DE L'INSTANCE : {instance_name}")
        print("=" * 65)
        
        reseau = charger_instance_cvrptw(txt_file)
        sol_opt = vrplib.read_solution(sol_file)
        score_optimal = sol_opt.get("cost", sol_opt.get("Cost"))
        camions_optimal = len(sol_opt.get("routes", []))
        
        print(f"🎯 Optimum théorique : {score_optimal:.2f} ({camions_optimal} camions)")
        
        # Historiques locaux pour cette instance
        hist_scores = {nom: [] for nom in noms_algos}
        hist_gaps = {nom: [] for nom in noms_algos}
        hist_camions = {nom: [] for nom in noms_algos}
        hist_temps = {nom: [] for nom in noms_algos}
        
        def enregistrer_run(seed_idx, nom_algo, solution, duree):
            score = reseau.calculate_objective(solution)
            gap = 100.0 * (score - score_optimal) / score_optimal
            nb_cam = len(solution)
            
            hist_scores[nom_algo].append(score)
            hist_gaps[nom_algo].append(gap)
            hist_camions[nom_algo].append(nb_cam)
            hist_temps[nom_algo].append(duree)
            
            writer_runs.writerow([
                instance_name, seed_idx, nom_algo, f"{score:.2f}", f"{score_optimal:.2f}",
                f"{gap:.2f}", nb_cam, camions_optimal, f"{duree:.4f}"
            ])

        # ==========================================
        # 4. BOUCLE SUR LES N SEEDS
        # ==========================================
        for seed in range(N_SEEDS):
            random.seed(seed)
            np.random.seed(seed)
            print(f"  ↳ Exécution Seed {seed + 1}/{N_SEEDS}...", end="\r")
            
            # 1. Aléatoire
            t0 = time.perf_counter()
            sol_alea = random_solution_multi(reseau)
            enregistrer_run(seed, 'Aléatoire', sol_alea, time.perf_counter() - t0)
            
            # 2. Plus Proche Voisin
            t0 = time.perf_counter()
            sol_nn = nearest_neighbor_solution_multi(reseau)
            enregistrer_run(seed, 'Plus Proche Voisin', sol_nn, time.perf_counter() - t0)
            
            # 3. Hill Climbing (depuis Plus Proche Voisin)
            t0 = time.perf_counter()
            sol_hc = hill_climbing_multi(sol_nn, reseau, methode_voisinage="relocate")
            enregistrer_run(seed, 'Hill Climbing', sol_hc, time.perf_counter() - t0)
            
            # 4. VND (depuis Plus Proche Voisin)
            t0 = time.perf_counter()
            sol_vnd = vnd_multi(sol_nn, reseau)
            enregistrer_run(seed, 'VND', sol_vnd, time.perf_counter() - t0)
            
            # 5. Tabou Search (Voisinage aléatoire depuis HC)
            t0 = time.perf_counter()
            sol_tabu = recherche_tabou_multi(sol_hc, reseau, taille_tabou=20, iter_max=500, methode_voisinage="random")
            enregistrer_run(seed, 'Tabou Search', sol_tabu, time.perf_counter() - t0)
            
            # 6. Simulated Annealing (Voisinage aléatoire depuis HC)
            t0 = time.perf_counter()
            sol_sa = recuit_simule_multi(sol_hc, reseau, temp_initiale=100.0, taux_refroidissement=0.95, iter_max=200, methode_voisinage="random")
            enregistrer_run(seed, 'Simulated Annealing', sol_sa, time.perf_counter() - t0)
            

            
            # 8. Algorithme Génétique
            t0 = time.perf_counter()
            sol_ga = algorithme_genetique_multi(sol_nn, reseau, pop_size=8, generations=15, methode_voisinage="random")
            enregistrer_run(seed, 'Algo Génétique', sol_ga, time.perf_counter() - t0)

        print(f"  ✅ {N_SEEDS} seeds terminées pour {instance_name}.              ")
        
        # Sauvegarde des moyennes et écarts-types dans le fichier stats
        for nom in noms_algos:
            writer_stats.writerow([
                instance_name, nom, N_SEEDS,
                f"{np.mean(hist_scores[nom]):.2f}", f"{np.std(hist_scores[nom]):.2f}",
                f"{score_optimal:.2f}",
                f"{np.mean(hist_gaps[nom]):.2f}", f"{np.std(hist_gaps[nom]):.2f}",
                f"{np.mean(hist_camions[nom]):.1f}", camions_optimal,
                f"{np.mean(hist_temps[nom]):.4f}", f"{np.std(hist_temps[nom]):.4f}"
            ])

print(f"\n💾 Export terminé :\n  - Détails : {FILE_RUNS_CSV}\n  - Stats   : {FILE_STATS_CSV}")

{'name': 'C101', 'vehicles': 25, 'capacity': 200, 'node_coord': array([[40, 50],
       [45, 68],
       [45, 70],
       [42, 66],
       [42, 68],
       [42, 65],
       [40, 69],
       [40, 66],
       [38, 68],
       [38, 70],
       [35, 66],
       [35, 69],
       [25, 85],
       [22, 75],
       [22, 85],
       [20, 80],
       [20, 85],
       [18, 75],
       [15, 75],
       [15, 80],
       [30, 50],
       [30, 52],
       [28, 52],
       [28, 55],
       [25, 50],
       [25, 52],
       [25, 55],
       [23, 52],
       [23, 55],
       [20, 50],
       [20, 55],
       [10, 35],
       [10, 40],
       [ 8, 40],
       [ 8, 45],
       [ 5, 35],
       [ 5, 45],
       [ 2, 40],
       [ 0, 40],
       [ 0, 45],
       [35, 30],
       [35, 32],
       [33, 32],
       [33, 35],
       [32, 30],
       [30, 30],
       [30, 32],
       [30, 35],
       [28, 30],
       [28, 35],
       [26, 32],
       [25, 30],
       [25, 35],
       [44,  5],
       [42, 10],
 

In [ ]:
import os
import csv
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

def generer_plots_depuis_csv(filepath="results/benchmark_stats_agregees.csv"):
    if not os.path.exists(filepath):
        print(f"❌ Fichier introuvable : {filepath}. Lancez d'abord le benchmark.")
        return

    donnees_par_instance = defaultdict(list)
    with open(filepath, mode='r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            donnees_par_instance[row["Instance"]].append(row)

    couleurs = ['#95a5a6', '#f39c12', '#2ecc71', '#1abc9c', '#3498db', '#9b59b6', '#e74c3c', '#34495e']

    for instance_name, rows in donnees_par_instance.items():
        algos = [r["Algorithm"] for r in rows]
        moy_scores = [float(r["Mean_Score"]) for r in rows]
        std_scores = [float(r["Std_Score"]) for r in rows]
        moy_temps = [float(r["Mean_Time_s"]) for r in rows]
        std_temps = [float(r["Std_Time_s"]) for r in rows]
        opt_score = float(rows[0]["Optimal_Score"])
        n_seeds = rows[0]["N_Seeds"]

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))
        fig.suptitle(f"Benchmark CVRPTW - Instance {instance_name} ({n_seeds} exécutions)", fontsize=16, fontweight='bold')

        # --- Graphique 1 : Score moyen ---
        bars1 = ax1.bar(
            algos, moy_scores, yerr=std_scores, capsize=5,
            edgecolor='black', color=couleurs[:len(algos)], error_kw={'elinewidth': 1.5, 'alpha': 0.8}
        )
        
        # 🟢 MODIFICATION ICI : Calcul de la cible à 10% (Optimum * 1.10)
        cible_10_pct = opt_score * 1.10
        ax1.axhline(cible_10_pct, color='red', linestyle='--', linewidth=2, label=f"Objectif Industriel (+10% Gap) : {cible_10_pct:.1f}")
        
        ax1.set_title("Coût Total Moyen (Distance / Temps)", fontsize=13, fontweight='bold')
        ax1.set_ylabel("Coût de la fonction objectif (Z)", fontsize=11)
        ax1.tick_params(axis='x', rotation=25)
        ax1.set_ylim(0, max([m + s for m, s in zip(moy_scores, std_scores)]) * 1.18)
        ax1.legend(loc="upper right", fontsize=11)
        ax1.grid(axis='y', linestyle=':', alpha=0.6)

        for i, bar in enumerate(bars1):
            yval = bar.get_height()
            ax1.text(
                bar.get_x() + bar.get_width() / 2, yval + std_scores[i] + (max(moy_scores) * 0.02),
                f"{yval:.0f}", ha='center', va='bottom', fontweight='bold', fontsize=9
            )

        # --- Graphique 2 : Temps d'exécution moyen ---
        bars2 = ax2.bar(
            algos, moy_temps, yerr=std_temps, capsize=5,
            edgecolor='black', color=couleurs[:len(algos)], error_kw={'elinewidth': 1.5, 'alpha': 0.8}
        )
        ax2.set_title("Temps d'Exécution Moyen", fontsize=13, fontweight='bold')
        ax2.set_ylabel("Temps (secondes)", fontsize=11)
        ax2.tick_params(axis='x', rotation=25)
        ax2.set_ylim(0, max([m + s for m, s in zip(moy_temps, std_temps)]) * 1.18)
        ax2.grid(axis='y', linestyle=':', alpha=0.6)

        for i, bar in enumerate(bars2):
            yval = bar.get_height()
            ax2.text(
                bar.get_x() + bar.get_width() / 2, yval + std_temps[i] + (max(moy_temps) * 0.02),
                f"{yval:.2f}s", ha='center', va='bottom', fontweight='bold', fontsize=9
            )

        plt.tight_layout()
        plt.show()

# Lancement de l'affichage
generer_plots_depuis_csv("results/benchmark_stats_agregees.csv")